# Text Generation - 소설 토지(박경리) 사용
- 단어(공백으로 구분) 단위 자연어 생성

In [1]:
import tensorflow as tf
import numpy as np
import re

path_to_file = tf.keras.utils.get_file(
    'toji.txt',
    'https://raw.githubusercontent.com/pykwon/python/refs/heads/master/testdata_utf8/rnn_test_toji.txt'
)

with open(path_to_file, encoding='utf-8', errors='ignore') as obj:
    raw_text = obj.read()

print(raw_text[:100])
print('문자 수 :', len(raw_text)) # 문자 수 : 677125

1588545/1588545 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
제 1 편 어둠의 발소리
1897년의 한가위.
까치들이 울타리 안 감나무에 와서 아침 인사를 하기도 전에, 무색 옷에 댕기꼬리를 늘인 
아이들은 송편을 입에 물고 마을길을 쏘다니며
문자 수 : 677125


In [2]:
def clean_str(text:str) -> str:
  text = re.sub(r"[^가-힣0-9() \n]", " ", text)
  text = re.sub(r"\s{2,}"," ",text)
  return text.strip()

# Call the function and process the text outside the function definition
cleand = clean_str(raw_text)
# print(cleand)
corpus = cleand.replace('\n',' [NL] ') # 줄바꿈을 토큰으로 처리하기 위해 특수문자 사용
# print(corpus)

# 토큰 처리 : 문자열 -> 단어분리 -> 단어사전 -> 정수번호로 변환
vectorizer = tf.keras.layers.TextVectorization(
    standardize=None,
    split='whitespace',
    output_mode='int',
    output_sequence_length=None,
    vocabulary=None
)

# 단어 사전 만들기
vectorizer.adapt(tf.data.Dataset.from_tensor_slices([corpus]))

vocab = vectorizer.get_vocabulary()
# print(vocab)
PAD, UNK = 0, 1 # 패딩, 알수 없는 값 상수값 정의하기
vocab_size = len(vocab)
print(f'어휘 수 : {vocab_size} (PAD={PAD}, UNK={UNK})')
print('샘플 어휘 :', vocab[:20])

token_ids = vectorizer(tf.constant([corpus]).numpy())[0] # 토큰 id 시퀀스
print(f'토큰 수 : {len(token_ids)}') # 토큰 수 : 164150
print(token_ids) # tf.Tensor([   51 51341  2059 ...
print(vocab[51],' ', vocab[51341],' ',vocab[2059])

if len(token_ids) <= 50:
  raise ValueError("토큰 수가 너무 적어 작업 안함")

# 학습용 시퀀스
SEQ_LEN = 20  # 입력 길이(과거 25개의 토큰을 보고 다음 토큰 예측)
BATCH = 256   # 배치 크기
BUFFER = 5000 # 셔플 버퍼

# tf.data.Dataset은 텐서플로우에서 고성능 데이터 입력 파이프라인을 구축하기 위한 핵심 API
# 메모리 내 데이터를 슬라이싱하요 데이터셋을 생성하고, map(), batch(), shuffle()등으로 전처리 한 뒤 반복 순회하여 사용

ds = tf.data.Dataset.from_tensor_slices(token_ids) # 배열이나 리스트를 한 개씩 잘라서 Dataset으로 만듦
ds = ds.window(SEQ_LEN + 1, shift=1, drop_remainder=True) # 한 칸씩 우측으로 밀기
ds = ds.flat_map(lambda w:w.batch(SEQ_LEN + 1)) # 각 윈도우 텐서로 수집
# Dataset 안의 각 원소를 다시 Dataset으로 바꾼뒤 하나로 펼치기
# 1 -> [1, 11]
# 2 -> [2, 12]
# 3 -> [3, 13] ===> 1 11 2 12 3 13

# 예측 함수에서 많이 쓰이는 함수
# 하나의 토큰 묶음(chunk)을 입력(x), 정답(y), 가중치(w)로 나누는 함수
def split_xyFunc(chunk): # chunk데이터는 SEQ_LEN + 1이 들어옴
  x = chunk[:-1]  # 입력 (마지막 값 제외)
  y = chunk[1:]   # 정답(Traget) - 각 시점의 다음 토큰 예측
  w = tf.cast(tf.not_equal(y, PAD), tf.float32) # 정답이 실제 token이면 1, 정답이 PAD면 0을 반환
  return x, y, w

#SEQ_LEN=5 이면 chunk=[10, 25, 31, 8, 47, 0]이라고 할때 len(chunk)=6
# x = [10, 25, 31, 8, 47]
# (정답) ↓  ↓   ↓   ↓   ↓  # 입력값에 다음값을 정답으로 학습을 함
# y = [25, 31, 8, 47, 0]

# 파이프라인 구축
ds = (ds.map(
    split_xyFunc,
    num_parallel_calls=tf.data.AUTOTUNE) # 학습 데이터 준비 병렬화 - 처리속도를 빠르게함
    .cache()               # 메모리 캐시 사용
    .shuffle(BUFFER)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE) # 학습 데이터 준비 병렬화
)
windows = len(token_ids) - SEQ_LEN # window의 크기
steps_per_epoch = max(1, windows // BATCH) # 0 방지
print('steps_per_epoch :', steps_per_epoch)

# 모델 정의하기
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQ_LEN,)),
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=128, mask_zero=True),
    tf.keras.layers.LSTM(256, return_sequences=True), # many2many - 기존글을 읽어서 새로운 글을 만들어야 하니까 softmax사용, 감성분류는 False
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(vocab_size) # , activation='softmax'
])

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)# loss를 직접 계산
model.compile(
    optimizer='adam',
    loss=loss_fn,
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy] # 정답 labeld이 onthot 인코딩이 아닌 경우 점수볼때 사용 할 수 있는 방법
 )
model.summary()

어휘 수 : 51358 (PAD=0, UNK=1)
샘플 어휘 : ['', '[UNK]', np.str_('[NL]'), np.str_('그'), np.str_('안'), np.str_('있었다'), np.str_('다'), np.str_('한'), np.str_('것'), np.str_('이'), np.str_('것이다'), np.str_('있는'), np.str_('수'), np.str_('없는'), np.str_('하고'), np.str_('못'), np.str_('같은'), np.str_('와'), np.str_('그러나'), np.str_('내')]
토큰 수 : 164150
tf.Tensor([   51 51341  2059 ...    49  1590   275], shape=(164150,), dtype=int64)
제   1   편
steps_per_epoch : 641


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 128)        │     6,573,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 20, 256)        │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 20, 256)        │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20, 51358)      │    13,199,006 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,232,862 (77.18 MB)

 Trainable params: 20,232,862 (77.18 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 모델이 예측한 점수인 logits를 확률로 바꾼 뒤, 그 확률에 따라 다음 토큰 하나를 추가하는 함수
def sample_from_logits(logits, temperature=1.0, top_k=0, forbid_ids=(0, 1)):
  # 순서1 : logits을 받음
    logits = logits.astype(np.float64)

  # forbid_ids : 생성하면 안되는 토큰 목록
  # 순서2 : PAD, UNK 같은 금지 토큰을 제외함
    for tid in forbid_ids:
        if 0 <= tid < logits.size:
            logits[tid] = -np.inf
        # 글자를 생성할 때 temperature를 주는데 얼마나 창의적인지를 조절할때 사용
        # temperature로 확률분포 조절 -> 0에 근사하면 보수적, 커지면 창의적
        if temperature <= 0:
            temperature = 1e-8
        logits = logits / temperature

        # top_k 가 있다면 상위 k개 후보만 남김
        if top_k:
            k = min(int(top_k), logits.size)
            if k > 0 and k < logits.size:
                idxs = np.argpartition(-logits, k)[:k]

                # 배열에서 값 자체를 정렬하는 함수가 아니라,
                # 특정 기준 위치에 올 원소들의 “인덱스”를 빠르게 찾는 함수다.
                # 특히 상위 K개 / 하위 K개 인덱스를 찾을 때 많이 사용한다.
                mask = np.full_like(logits, -np.inf)
                mask[idxs] = logits[idxs]
                logits = mask

        logits = logits - np.max(logits) # softmax로 확률을 만듦
        probs = np.exp(logits)
        probs_sum = probs.sum()
        if probs_sum == 0 or not np.isfinite(probs_sum):   # 전부 -inf가 되는 특이 케이스 방어
            probs = np.ones_like(probs) / probs.size
        else:
            probs /= probs_sum

        return np.random.choice(len(probs), p=probs)  # 확률에 따라 샘플 하나 선택

# 주기적으로 샘플을 생성
idx2tok = np.array(vocab, dtype=object) # 단어 사전 리스트(vocab)를 numpy배열로 반환

# 토큰 id를 사람이 읽을 수 있는 문장으로 반환하는 함수
def ids_to_text(ids):   # ex) ids=[2, 3, 5] -> ['사람','간다','나는'] 이런 역할을 하는 함수
  toks = idx2tok[ids]
  toks = [("\n" if t == "[NL]" else t) for t in toks] # \n을[NL]로 바꾼걸 다시 \n으로 바꿈 - [NL]토큰을 개행(\n)으로 복원
  return " ".join(toks).replace(" \n ","\n").replace("\n ","\n").replace(" \n","\n")

# 사용자가 넣은 시작 문장을 바탕으로 학습된 모델이 뒤에 이어질 문장으로 자동 생성하는 함수
def generateFunc(seed_text, max_new_tokens=80, temperature=0.9, top_k=30):
  seed = clean_str(seed_text).replace('\n',' [NL] ')
  seed_ids = vectorizer(tf.constant([seed])).numpy()[0].tolist()
  # context 길이를 SEQ_LEN으로 맞춤(왼쪽은 PAD로 채움(공백)) -> RNN 입력 길이를 고정
  context = [PAD] * max(0, SEQ_LEN,- len(seed_ids)) + seed_ids[-SEQ_LEN:]

  out_ids = [] # 생성된 token id 누적용 list
  for _ in range(max_new_tokens):
    x = np.array(context, dtype=np.int32)[None, :] # (1, SEQ_LEN) 배치화
    logits = model.predcit(x, verbose=0)[0, -1] # 마지막 시점의 로짓만 사용
    tid = sample_from_logits(
                  logits,
                  temperature=temperature,
                  top_k=top_k,
                  forbid_ids=(PAD, UNK)
        )

    out_ids.append(tid)
    context = context[1:] + [tid] # context에 방금 샘플 추가-> 윈도우 우측 이동

  text = ids_to_text(out_ids)
  text = re.sub(r"\s{2,}", text).strip() # tab, 공백이 중복이면 중복 처리
  return text

# 일정 주기로 샘플 출력 - 학습 진행 상태 확인용 클래스
class SamplerCallback(tf.keras.callbacks.Callback):
  # Callback 오버라이딩 : 매 epoch이 끝날때 마다 자동 호출
  def on_each_end(self, epoch, logs=None):
    # 5 epoch마다 마지막 epoch은 항상 출력
    if epoch % 5 != 0 and epoch != (self.params.get('epochs',1) - 1):
      return

    # 비교용 고정 seed
    seed = "아이들은 송편을 입에 물고 마을길을 쏘다니며"
    sample = generateFunc(seed, max_new_tokens=80, temperature=0.9, top_k=30)
    print("\n[샘플 생성]", epoch)
    print(seed + " " + sample[:500])


EPOCHS = 2 # 원래는 많이 줘야 함(오래걸림)
history = model.fit(ds,
                    epochs=EPOCHS,
                    steps_per_epoch=steps_per_epoch,
                    callbacks=[SamplerCallback()]
          )
print('final loss :', float(history.history['loss'][-1]))
print('final accuracy :', float(history.history['sparse_categorical_accuracy'][-1]))

# 모델 정의하기 - 수정된 부분
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQ_LEN,)),
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=128, mask_zero=True),
    tf.keras.layers.LSTM(128, return_sequences=True), # Reduced units from 256 to 128
    tf.keras.layers.Dense(128, activation='relu'),   # Reduced units from 256 to 128
    tf.keras.layers.Dense(vocab_size)
])

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(
    optimizer='adam',
    loss=loss_fn,
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy]
 )
model.summary()


# 최종 테스트
seed = '아이들은 송편을 입에 물고 마을길을 쏘다니며'
out = generateFunc(seed,
                   max_new_tokens=200,
                   temperature=0.8,
                   top_k=40
      )
print('최종 결과 : \n')
print(seed, out)

Epoch 1/2
